# M3 — Uji seed 123 dan 456 pada sepuluh gambar yang sama

In [ ]:
# Jalankan pada Google Colab; paket dipasang hanya di runtime Colab.
%pip install -q open_clip_torch

import csv
import hashlib
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4
from zipfile import ZipFile

import numpy as np
import open_clip
import torch
from PIL import Image
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/Dataset-Tubes-Multimodal")
ZIP = DRIVE / "bundle-mahasiswa.zip"
DATA_DIR = Path("/content/data/bundle-mahasiswa")
SEEDS = (42, 123, 456)  # Seed 
CHECKPOINT_DIR = DRIVE / "results/m2/e1p_rerun_20261120T134751Z_61ea1f37"
CHECKPOINTS = {seed: CHECKPOINT_DIR / f"projection_best_seed_{seed}.pt" for seed in SEEDS}
TEMPERATURE = 0.07  

if not ZIP.is_file():
    raise FileNotFoundError(f"Bundel dataset belum ada di Drive: {ZIP}")
for seed, checkpoint in CHECKPOINTS.items():
    if not checkpoint.is_file():
        raise FileNotFoundError(f"Checkpoint seed {seed} belum ada di Drive: {checkpoint}")
if not (DATA_DIR / "val.csv").is_file():
    Path("/content/data").mkdir(parents=True, exist_ok=True)
    with ZipFile(ZIP) as bundle:
        bundle.extractall("/content/data")  # ZIP milik proyek, bukan URL tak tepercaya.
if not (DATA_DIR / "val.csv").is_file():
    raise FileNotFoundError(f"val.csv tidak ditemukan setelah ekstraksi: {DATA_DIR}")
print("Data validasi dan checkpoint seed 123/456 ditemukan; belum ada prediksi dijalankan.")


In [ ]:
# FUNGSI_M3_MURNI
# SEA-VL Indonesian subset: KORIKA-AI/sea-vl_crowdsourcing_id,
# turunan SEACrowd/sea-vl_crowdsourcing; CC BY-SA 4.0.
# Cahyawijaya dkk., "Crowdsource, Crawl, or Generate?", ACL 2025.
import os
import math


CASES = [
    {"case_id": "A1", "group": "A", "image_id": "svl_142112996", "focus": "negasi",
     "hard": ("Tiang bendera tanpa bendera.", "Tiang bendera dengan bendera."),
     "control": ("Foto tiang bendera.", "Foto menara candi.")},
    {"case_id": "A2", "group": "A", "image_id": "svl_140084661", "focus": "pose",
     "hard": ("Tiga penari berbaju kuning duduk.", "Tiga penari berbaju kuning berdiri."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A3", "group": "A", "image_id": "svl_140084668", "focus": "warna",
     "hard": ("Penari berbaju putih berdiri.", "Penari berbaju kuning berdiri."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A4", "group": "A", "image_id": "svl_141004074", "focus": "spasial",
     "hard": ("Pedang di kiri foto, perisai di kanan foto.",
              "Pedang di kanan foto, perisai di kiri foto."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "A5", "group": "A", "image_id": "svl_140084373", "focus": "benda-budaya",
     "hard": ("Kemeja batik di atas meja.", "Taplak batik menutupi meja."),
     "control": ("Foto kemeja batik.", "Foto menara candi.")},
    {"case_id": "B1", "group": "B", "image_id": "svl_140084155", "focus": "alat-makan",
     "hard": ("Bakmi di piring dengan sumpit hitam.",
              "Bakmi di piring dengan sendok hijau."),
     "control": ("Foto sepiring bakmi.", "Foto stupa candi.")},
    {"case_id": "B2", "group": "B", "image_id": "svl_140084156", "focus": "alat-makan",
     "hard": ("Bakmi di piring dengan sendok hijau.",
              "Bakmi di piring dengan sumpit hitam."),
     "control": ("Foto sepiring bakmi.", "Foto stupa candi.")},
    {"case_id": "B3", "group": "B", "image_id": "svl_140084408", "focus": "benda-tarian",
     "hard": ("Tiga penari mengangkat selendang jingga.",
              "Tiga penari mengangkat payung jingga."),
     "control": ("Foto para penari.", "Foto stupa candi.")},
    {"case_id": "B4", "group": "B", "image_id": "svl_143496795", "focus": "arsitektur-budaya",
     "hard": ("Stupa Borobudur berbentuk lonceng.",
              "Menara Prambanan berujung runcing."),
     "control": ("Foto stupa candi.", "Foto para penari.")},
    {"case_id": "B5", "group": "B", "image_id": "svl_144425377", "focus": "arsitektur-budaya",
     "hard": ("Menara Prambanan berujung runcing.",
              "Stupa Borobudur berbentuk lonceng."),
     "control": ("Foto menara candi.", "Foto sepiring bakmi.")},
]


def validasi_kasus(cases: list[dict], rows: list[dict], root: Path) -> list[dict]:
    """Pastikan gambar unik, berasal dari val, dan berkasnya tersedia.

    Args:
        cases: Sepuluh kasus berlabel dari rencana M3.
        rows: Baris val.csv yang sudah dibaca.
        root: Direktori induk val.csv dan images/.
    Returns:
        Baris metadata val.csv sesuai urutan kasus.
    Raises:
        ValueError: Jika kasus/metadata tidak valid atau berduplikat.
        FileNotFoundError: Jika gambar tidak ditemukan.
    """
    if len(cases) != 10 or len({c["image_id"] for c in cases}) != 10:
        raise ValueError("Perlu 10 kasus dengan image_id unik, tanpa duplikat.")
    by_id = {row["image_id"]: row for row in rows}
    if len(by_id) != len(rows):
        raise ValueError("Duplikat image_id pada val.csv.")
    selected = []
    for case in cases:
        if case["group"] not in {"A", "B"}:
            raise ValueError("Kelompok kasus harus A atau B.")
        for kind in ("hard", "control"):
            texts = case[kind]
            if len(texts) != 2 or not all(isinstance(t, str) and t.strip() for t in texts) or texts[0] == texts[1]:
                raise ValueError(f"Kandidat {kind} tidak valid: {case['case_id']}")
        row = by_id.get(case["image_id"])
        if row is None or row.get("split") != "val":
            raise ValueError(f"Gambar bukan dari val.csv: {case['image_id']}")
        file = root / row["file"]
        if not file.is_file():
            raise FileNotFoundError(f"Gambar tidak ditemukan: {file}")
        selected.append(row)
    return selected


def periksa_berkas_awal(dataset_zip: Path, checkpoints: dict[int, Path]) -> None:
    """Periksa keberadaan dataset dan kedua checkpoint sebelum inferensi.

    Args:
        dataset_zip: Bundel data di Google Drive.
        checkpoints: Path checkpoint untuk seed 123 dan 456.
    Returns:
        Tidak ada.
    Raises:
        ValueError: Bila pilihan seed tidak tepat.
        FileNotFoundError: Bila salah satu input belum tersedia.
    """
    if set(checkpoints) != {123, 456}:
        raise ValueError("Preflight membutuhkan checkpoint seed 123 dan 456.")
    if not dataset_zip.is_file():
        raise FileNotFoundError(f"Bundel dataset tidak ada: {dataset_zip}")
    for seed, checkpoint in checkpoints.items():
        if not checkpoint.is_file():
            raise FileNotFoundError(f"Checkpoint seed {seed} tidak ada: {checkpoint}")


class ProyeksiResidual(torch.nn.Module):
    """Proyeksi teks residual identik dengan definisi M2."""

    def __init__(self, dim: int) -> None:
        """Bangun layer W yang bentuknya sesuai checkpoint M2.

        Args:
            dim: Dimensi embedding SigLIP 2.
        Returns:
            Tidak ada.
        Raises:
            ValueError: Bila dimensi tidak positif.
        """
        if dim <= 0:
            raise ValueError("Dimensi proyeksi harus positif.")
        super().__init__()
        self.W = torch.nn.Linear(dim, dim, bias=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Terapkan residual dan normalisasi seperti M2.

        Args:
            x: Embedding teks dengan dimensi terakhir sama dengan layer W.
        Returns:
            Tensor embedding teks yang dinormalisasi.
        Raises:
            RuntimeError: Bila dimensi tensor tidak kompatibel.
        """
        output = x + self.W(x)
        return output / output.norm(dim=-1, keepdim=True)


def muat_proyeksi(checkpoint: Path, expected_dim: int, expected_seed: int) -> ProyeksiResidual:
    """Muat checkpoint tepercaya di CPU dan validasi metadata.

    Args:
        checkpoint: Path file .pt hasil run M2.
        expected_dim: Dimensi teks SigLIP 2 yang terukur saat inferensi.
        expected_seed: Seed yang dipilih untuk kasus M3.
    Returns:
        ProyeksiResidual siap inferensi.
    Raises:
        FileNotFoundError: Jika checkpoint tidak tersedia.
        ValueError: Jika encoder, seed, atau dimensi berbeda.
        RuntimeError: Jika state_dict tidak sesuai dengan layer.
    """
    if expected_dim <= 0 or expected_seed not in (42, 123, 456):
        raise ValueError("Dimensi embedding tidak valid.")
    if not Path(checkpoint).is_file():
        raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint}")
    payload = torch.load(checkpoint, map_location="cpu", weights_only=True)
    expected = {"format_version": 1, "encoder": "SigLIP 2 Base",
                "encoder_model": "ViT-B-16-SigLIP2-256", "encoder_pretrained": "webli",
                "seed": expected_seed, "dim": expected_dim}
    if not isinstance(payload, dict) or any(payload.get(k) != v for k, v in expected.items()):
        raise ValueError(f"Metadata checkpoint tidak cocok dengan E1+P seed {expected_seed}: {checkpoint}")
    model = ProyeksiResidual(expected_dim)
    model.load_state_dict(payload["state_dict"], strict=True)
    model.eval()
    return model


def skor_pasangan(image: torch.Tensor, texts: torch.Tensor,
                 proj: ProyeksiResidual) -> tuple[float, float]:
    """Hitung kosinus antara gambar dan dua embedding teks proyeksi.

    Args:
        image: Tensor embedding gambar (1, dim).
        texts: Tensor embedding dua kandidat teks (2, dim).
        proj: Layer proyeksi teks seed 42.
    Returns:
        Dua skor kemiripan dalam urutan kandidat sesuai/pengecoh.
    Raises:
        ValueError: Jika dimensi atau nilai embedding tidak valid.
    """
    if image.ndim != 2 or texts.ndim != 2 or image.shape[0] != 1 or texts.shape[0] != 2 or image.shape[1] != texts.shape[1]:
        raise ValueError("Harus ada satu gambar dan dua teks dengan dimensi sama.")
    with torch.no_grad():
        norm_image = image / image.norm(dim=-1, keepdim=True)
        projected = proj(texts)
        similarity = (norm_image @ projected.T).squeeze(0)
    if not torch.isfinite(similarity).all():
        raise ValueError("Skor non-finite: periksa embedding/model.")
    return tuple(float(v) for v in similarity.tolist())


def probabilitas_dua_kandidat(scores: tuple[float, float],
                              temperature: float) -> tuple[float, float]:
    """Softmax relatif untuk tepat dua teks, bukan probabilitas kalibrasi.

    Args:
        scores: Dua skor kosinus dalam urutan A lalu B.
        temperature: Skala yang ditetapkan sebelum inferensi.
    Returns:
        Probabilitas relatif A dan B yang jumlahnya satu.
    Raises:
        ValueError: Jika skala atau skor tidak valid.
    """
    if len(scores) != 2 or not math.isfinite(temperature) or temperature <= 0 or not all(math.isfinite(s) for s in scores):
        raise ValueError("Softmax membutuhkan dua skor finite dan temperature positif.")
    scaled = [s / temperature for s in scores]
    peak = max(scaled)
    exp = [math.exp(v - peak) for v in scaled]
    total = sum(exp)
    return exp[0] / total, exp[1] / total


def baris_hasil(case: dict, kind: str, scores: tuple[float, float],
               temperature: float) -> list[dict]:
    """Susun dua baris hasil tanpa menyembunyikan kandidat pengecoh/seri.

    Args:
        case: Kasus M3 berisi kandidat sesuai dan pengecoh.
        kind: 'hard' atau 'control'.
        scores: Kosinus dua teks sesuai urutan case[kind].
        temperature: Skala softmax relatif.
    Returns:
        Dua baris kandidat dengan skor, probabilitas, label, dan status.
    Raises:
        ValueError: Jika jenis kasus tidak dikenali.
    """
    if kind not in ("hard", "control"):
        raise ValueError("Jenis hasil harus hard atau control.")
    probs = probabilitas_dua_kandidat(scores, temperature)
    passed = scores[0] > scores[1]  # Seri dihitung gagal, tidak dipecahkan berdasarkan urutan.
    return [{"case_id": case["case_id"], "group": case["group"],
             "image_id": case["image_id"], "focus": case["focus"],
             "test_type": kind, "candidate_label": label, "candidate_text": text,
             "cosine_score": score, "probabilitas_relatif": prob,
             "temperature": temperature, "lolos": passed}
            for text, label, score, prob in zip(
                case[kind], ("sesuai", "pengecoh"), scores, probs)]


In [ ]:
# Kunci kasus dan cek dua checkpoint serta seluruh gambar sebelum inferensi.
periksa_berkas_awal(ZIP, CHECKPOINTS)
with (DATA_DIR / "val.csv").open(encoding="utf-8-sig", newline="") as handle:
    val_rows = list(csv.DictReader(handle))
selected = validasi_kasus(CASES, val_rows, DATA_DIR)
print(f"{len(selected)} gambar val unik; A=5, B=5; seluruh kandidat tetap.")
for case, row in zip(CASES, selected):
    print(case["case_id"], row["image_id"], case["focus"], row["file"])


In [ ]:
# Load E1 beku dengan konfigurasi encoder M2; tidak ada train/fine-tuning.
device = "cuda" if torch.cuda.is_available() else "cpu"
encoder, _, preprocess = open_clip.create_model_and_transforms(
    "ViT-B-16-SigLIP2-256", pretrained="webli"
)
tokenizer = open_clip.get_tokenizer("ViT-B-16-SigLIP2-256")
encoder = encoder.eval().to(device)
with torch.no_grad():
    sample = tokenizer([CASES[0]["hard"][0]]).to(device)
    dim = int(encoder.encode_text(sample).shape[-1])
results_by_seed = {}
for seed in SEEDS:
    projection = muat_proyeksi(CHECKPOINTS[seed], expected_dim=dim, expected_seed=seed)
    results = []
    for case, row in zip(CASES, selected):
        with Image.open(DATA_DIR / row["file"]) as source:
            image_input = preprocess(source.convert("RGB")).unsqueeze(0).to(device)
        with torch.no_grad():
            image_embedding = encoder.encode_image(image_input).float().cpu()
            for kind in ("hard", "control"):
                tokens = tokenizer(list(case[kind])).to(device)
                text_embedding = encoder.encode_text(tokens).float().cpu()
                # M2 menormalisasi teks sebelum proyeksi.
                text_embedding = text_embedding / text_embedding.norm(dim=-1, keepdim=True)
                scores = skor_pasangan(image_embedding, text_embedding, projection)
                results.extend(baris_hasil(case, kind, scores, TEMPERATURE))
        print(seed, case["case_id"], "hard_lolos=", results[-4]["lolos"],
              "control_lolos=", results[-2]["lolos"])
    if len(results) != 40:
        raise ValueError(f"Seed {seed}: diharapkan 40 baris kandidat, ada {len(results)}")
    results_by_seed[seed] = results
    print(f"Skoring seed {seed} selesai; 40 kandidat tersimpan di memori.")


In [ ]:
# Tulis seluruh bukti ke folder run M3 BARU; tidak mengubah hasil M2.
RUN_ROOT = DRIVE / "results/m3"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def sha256_file(path: Path) -> str:
    """Hitung SHA-256 berkas agar asal artefak dapat diverifikasi.

    Args:
        path: Berkas lokal atau Drive yang dapat dibaca.
    Returns:
        Hash SHA-256 heksadesimal.
    Raises:
        OSError: Bila berkas tidak dapat dibaca.
    """
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


for seed in SEEDS:
    results = results_by_seed[seed]
    checkpoint = CHECKPOINTS[seed]
    metadata = {
        "model": "SigLIP 2 Base + ProyeksiResidual",
        "encoder_id": "ViT-B-16-SigLIP2-256",
        "encoder_pretrained": "webli", "seed": seed,
        "checkpoint_path": str(checkpoint),
        "checkpoint_sha256": sha256_file(checkpoint),
        "val_csv_sha256": sha256_file(DATA_DIR / "val.csv"),
        "split": "val", "temperature": TEMPERATURE,
        "probability_definition": "softmax atas tepat dua kandidat teks per gambar, tidak terkalibrasi",
        "tie_rule": "skor seri = gagal",
        "normalization": "cosine(image, projection(L2(text)))",
        "open_clip_torch_version": version("open_clip_torch"),
        "torch_version": torch.__version__,
        "case_images": {case["case_id"]: {"image_id": row["image_id"],
                         "sha256": sha256_file(DATA_DIR / row["file"])}
                        for case, row in zip(CASES, selected)},
        "cases": CASES,
    }
    run_dir = RUN_ROOT / f"m3_seed{seed}_{datetime.now(timezone.utc):%Y%m%dT%H%M%SZ}_{uuid4().hex[:8]}"
    run_dir.mkdir(exist_ok=False)
    with (run_dir / "hasil_kandidat.csv").open("x", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(results[0]))
        writer.writeheader()
        writer.writerows(results)
    with (run_dir / "metadata.json").open("x", encoding="utf-8") as handle:
        json.dump(metadata, handle, indent=2, ensure_ascii=False)
    print(f"Folder hasil seed {seed}: {run_dir}")
    for group in ("A", "B"):
        for kind in ("hard", "control"):
            subset = [r for r in results if r["group"] == group
                      and r["test_type"] == kind and r["candidate_label"] == "sesuai"]
            print(f"Seed {seed}, kelompok {group}, {kind}: {sum(r['lolos'] for r in subset)}/5 lolos")
print("Probabilitas relatif dan skor setiap kandidat ada di hasil_kandidat.csv; cek kegagalan sebelum menulis kesimpulan.")
